# StylistAI CatVTON Colab service

This notebook prepares CatVTON on a Google Colab GPU and starts the authenticated API used by the StylistAI local backend.

## Before running

1. Select **Runtime ? Change runtime type ? T4 GPU** (or another available GPU).
2. Run the setup cell once per fresh runtime and wait for the GPU/import check to complete.
3. Run the API cell. Enter your ngrok auth token only in the hidden prompt; it is not stored in this notebook source.
4. Copy the printed tunnel URL and generated API key into the local `backend/.env`, then restart the backend.

The API key is printed in the cell output so the local backend can be configured. Do not share or commit executed outputs. Clear outputs before saving notebook changes to GitHub. Colab GPU sessions and free ngrok tunnels are temporary.

CatVTON upstream: https://github.com/Zheng-Chong/CatVTON


## 1. Install dependencies and verify GPU


In [ ]:
%cd /content
!python -m pip install -q uv pyngrok
!uv python install 3.11
!uv venv /content/catvton-env311 --python 3.11
!if [ ! -d /content/CatVTON/.git ]; then git clone --depth 1 https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON; fi
!uv pip install --python /content/catvton-env311/bin/python torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/catvton-env311/bin/python "accelerate>=1.0.0" matplotlib==3.9.1 numpy==1.26.4 opencv_python==4.10.0.84 pillow==10.3.0 PyYAML==6.0.1 scipy==1.13.1 setuptools==51.0.0 scikit-image==0.24.0 tqdm==4.66.4 transformers==4.46.3 fvcore==0.1.5.post20221221 cloudpickle==3.0.0 omegaconf==2.3.0 pycocotools==2.0.8 av==12.3.0 peft==0.17.0 "huggingface_hub>=0.34.0,<2.0"
!uv pip install --python /content/catvton-env311/bin/python git+https://github.com/huggingface/diffusers.git@v0.30.3 gradio==4.44.1 gradio-client==1.3.0 fastapi==0.115.6 pydantic==2.10.6 uvicorn==0.34.0 python-multipart==0.0.20
!cd /content/CatVTON && /content/catvton-env311/bin/python -c "import torch; assert torch.cuda.is_available(), 'Connect a GPU runtime first'; from model.pipeline import CatVTONPipeline"
print("CatVTON setup complete. Next, run the API cell below.")


## 2. Start the API tunnel

This loads CatVTON in a separate API process without starting the original Gradio interface. The `/health` route is checked locally before the ngrok tunnel is opened. API requests require bearer authentication.


In [ ]:
import getpass, os, secrets, subprocess, time, urllib.error, urllib.request
from pathlib import Path

server_code = r'''import hmac, io, os, runpy, sys, threading
from PIL import Image, UnidentifiedImageError
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from starlette.concurrency import run_in_threadpool
os.environ.pop("MPLBACKEND", None)
os.chdir("/content/CatVTON")
sys.path.insert(0, os.getcwd())
sys.argv = ["catvton_api", "--output_dir=/content/CatVTON/resource/demo/output", "--mixed_precision=fp16", "--allow_tf32"]
ctx = runpy.run_path("app.py", run_name="catvton_api")
args, pipeline = ctx["args"], ctx["pipeline"]
automasker, mask_processor = ctx["automasker"], ctx["mask_processor"]
API_KEY = os.environ["CATVTON_API_KEY"]
app = FastAPI()
lock = threading.Lock()

def read_image(data):
    if not data or len(data) > 10 * 1024 * 1024:
        raise HTTPException(400, "Image is empty or exceeds 10 MB")
    try:
        image = Image.open(io.BytesIO(data))
        if image.width * image.height > 30_000_000:
            raise HTTPException(400, "Image dimensions are too large")
        return image.convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as exc:
        raise HTTPException(400, "Invalid image file") from exc

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/try-on")
async def try_on(person_image: UploadFile = File(...), cloth_image: UploadFile = File(...), cloth_type: str = Form(...), authorization: str = Header(default="")):
    if not hmac.compare_digest(authorization, "Bearer " + API_KEY):
        raise HTTPException(401, "Invalid API key")
    if cloth_type not in {"upper", "lower", "overall"}:
        raise HTTPException(400, "Unsupported garment type")
    person = read_image(await person_image.read())
    cloth = read_image(await cloth_image.read())
    def generate():
        with lock:
            person_resized = ctx["resize_and_crop"](person, (args.width, args.height))
            cloth_resized = ctx["resize_and_padding"](cloth, (args.width, args.height))
            mask = mask_processor.blur(automasker(person_resized, cloth_type)["mask"], blur_factor=9)
            result = pipeline(image=person_resized, condition_image=cloth_resized, mask=mask, num_inference_steps=50, guidance_scale=2.5)[0]
            output = io.BytesIO()
            result.save(output, format="PNG")
            return output.getvalue()
    return Response(content=await run_in_threadpool(generate), media_type="image/png")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''
Path('/content/catvton_api.py').write_text(server_code, encoding='utf-8')
api_key = secrets.token_urlsafe(32)
ngrok_token = getpass.getpass('Enter your ngrok auth token (input is hidden): ')
env = os.environ.copy()
env['CATVTON_API_KEY'] = api_key
env.pop('MPLBACKEND', None)
log_file = open('/content/catvton-api.log', 'w', encoding='utf-8')
server_process = subprocess.Popen(['/content/catvton-env311/bin/python', '/content/catvton_api.py'], cwd='/content/CatVTON', env=env, stdout=log_file, stderr=subprocess.STDOUT)
for _ in range(900):
    if server_process.poll() is not None:
        tail = Path('/content/catvton-api.log').read_text(encoding='utf-8', errors='replace').splitlines()[-60:]
        raise RuntimeError('CatVTON API failed to start:\n' + '\n'.join(tail))
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            if response.status == 200:
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise TimeoutError('CatVTON did not become healthy within 15 minutes; inspect /content/catvton-api.log')
from pyngrok import ngrok
ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, 'http')
print('CATVTON_API_URL=' + tunnel.public_url)
print('CATVTON_API_KEY=' + api_key)
print('Keep this notebook restricted; copy these values into the backend environment.')
